# Comparison

## Loading Dataset

In [ ]:
from datasets import load_dataset
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
from config import DATASET_NAME

dataset = load_dataset(DATASET_NAME)

train_samples = [{"image": x["image"], "text": x["text"]} for x in dataset["train"]]
val_samples = [{"image": x["image"], "text": x["text"]} for x in dataset["val"]]
test_samples = [{"image": x["image"], "text": x["text"]} for x in dataset["test"]]

print("Successfully loaded dataset")
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

## Pretrained vs Fine-tuned Predictions

In [ ]:
from PIL import Image, ImageOps

def predict(model, processor, sample, path=False):
    image = sample["image"].convert("RGB")
    image = ImageOps.grayscale(image)
    image = ImageOps.autocontrast(image)
    image = image.convert("RGB")

    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch
from config import PT_MODEL_NAME, FT_MODEL_NAME

pretrained_processor = TrOCRProcessor.from_pretrained(PT_MODEL_NAME)
pretrained_model = VisionEncoderDecoderModel.from_pretrained(PT_MODEL_NAME)
pretrained_model.eval()

finetuned_processor = TrOCRProcessor.from_pretrained(FT_MODEL_NAME)
finetuned_model = VisionEncoderDecoderModel.from_pretrained(FT_MODEL_NAME)
finetuned_model.eval()

### Runs each model on the TEST dataset

In [ ]:
from jiwer import cer, wer

pretrained_predictions = []
finetuned_predictions  = []
ground_truths = []


for i, sample in enumerate(test_samples):
    pt_pred = predict(pretrained_model, pretrained_processor, sample)
    ft_pred = predict(finetuned_model,  finetuned_processor,  sample)
    
    pretrained_predictions.append(pt_pred)
    finetuned_predictions.append(ft_pred)
    ground_truths.append(sample["text"])
    
    print(f"{i+1}/51 done")

print("\nPretrained — CER:", f"{cer(ground_truths, pretrained_predictions):.2%}", "| WER:", f"{wer(ground_truths, pretrained_predictions):.2%}")
print("Fine-tuned  — CER:", f"{cer(ground_truths, finetuned_predictions):.2%}",  "| WER:", f"{wer(ground_truths, finetuned_predictions):.2%}")

## Save predictions as JSON

In [ ]:
import json

results = {
    "ground_truths":          ground_truths,
    "pretrained_predictions": pretrained_predictions,
    "finetuned_predictions":  finetuned_predictions,
}

with open("../results/metrics/test_predictions.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to results/metrics/test_predictions.json")

## Load predictions from JSON

In [ ]:
import json

with open("../results/metrics/test_predictions.json", "r") as f:
    results = json.load(f)

ground_truths = results["ground_truths"]
pretrained_predictions = results["pretrained_predictions"]
finetuned_predictions = results["finetuned_predictions"]

print(f"Loaded {len(ground_truths)} samples")

## Prediction Viz

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = {
    "Pretrained": {"CER": cer(ground_truths, pretrained_predictions), "WER": wer(ground_truths, pretrained_predictions)},
    "Fine-tuned":  {"CER": cer(ground_truths, finetuned_predictions),  "WER": wer(ground_truths, finetuned_predictions)},
}

x     = np.arange(2)
width = 0.35
cer_vals = [metrics["Pretrained"]["CER"], metrics["Fine-tuned"]["CER"]]
wer_vals = [metrics["Pretrained"]["WER"], metrics["Fine-tuned"]["WER"]]

fig, ax = plt.subplots(figsize=(7, 5))
bars1 = ax.bar(x - width/2, [v * 100 for v in cer_vals], width, label="CER", color="#4C72B0")
bars2 = ax.bar(x + width/2, [v * 100 for v in wer_vals], width, label="WER", color="#DD8452")

ax.set_xticks(x)
ax.set_xticklabels(["Pretrained", "Fine-tuned"], fontsize=12)
ax.set_ylabel("Error Rate (%)", fontsize=12)
ax.set_title("Pretrained vs Fine-tuned: CER and WER", fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, max(max(wer_vals), max(cer_vals)) * 140)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("../results/figures/cer_wer_bar_chart.png", dpi=150)
plt.show()
print("Saved to results/figures/cer_wer_bar_chart.png")

In [ ]:
from jiwer import cer as sample_cer
import math

# Compute per-sample CER improvement
improvements = []
for i, (gt, pt, ft) in enumerate(zip(ground_truths, pretrained_predictions, finetuned_predictions)):
    pt_cer = sample_cer([gt], [pt])
    ft_cer = sample_cer([gt], [ft])
    improvement = pt_cer - ft_cer
    improvements.append((improvement, i, gt, pt, ft))

# Top 6 most improved
improvements.sort(reverse=True)
top = improvements[:6]

fig, axes = plt.subplots(6, 1, figsize=(12, 20))
fig.suptitle("Most Improved Predictions (Pretrained → Fine-tuned)", fontsize=14, y=1.01)

for ax, (improvement, i, gt, pt, ft) in zip(axes, top):
    sample = test_samples[i]
    image = sample["image"].convert("RGB")
    ax.imshow(image, cmap="gray")
    ax.axis("off")
    ax.set_title(
        f"GT:         {gt}\n"
        f"Pretrained: {pt}\n"
        f"Fine-tuned: {ft}\n"
        f"CER improvement: {improvement:.2%}",
        fontsize=9, loc="left", family="monospace"
    )

plt.tight_layout()
plt.savefig("../results/figures/most_improved.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to results/figures/most_improved.png")

In [ ]:
from jiwer import cer as sample_cer

pt_cers = [sample_cer([gt], [pt]) * 100 for gt, pt in zip(ground_truths, pretrained_predictions)]
ft_cers = [sample_cer([gt], [ft]) * 100 for gt, ft in zip(ground_truths, finetuned_predictions)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, cers, label, color in zip(
    axes,
    [pt_cers, ft_cers],
    ["Pretrained", "Fine-tuned"],
    ["#4C72B0", "#DD8452"]
):
    ax.hist(cers, bins=20, color=color, edgecolor="white", alpha=0.9)
    ax.axvline(np.mean(cers), color="black", linestyle="--", linewidth=1.5, label=f"Mean: {np.mean(cers):.1f}%")
    ax.set_title(f"{label} — Per-Sample CER Distribution", fontsize=12)
    ax.set_xlabel("CER (%)", fontsize=11)
    ax.set_ylabel("Number of Samples", fontsize=11)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("../results/figures/per_sample_cer_histogram.png", dpi=150)
plt.show()
print("Saved to results/figures/per_sample_cer_histogram.png")

## Training Process Viz

In [ ]:
import json
import matplotlib.pyplot as plt

with open("../results/metrics/lr=1e-5_augTrue_15ep.json", "r") as f:
    best_history = json.load(f)

epochs = range(1, len(best_history["train_loss"]) + 1)

fig, ax2 = plt.subplots(figsize=(8, 5))
ax3 = ax2.twinx()

line1, = ax2.plot(epochs, best_history["train_loss"], color="#4C72B0", marker="o", label="Train Loss")
line2, = ax3.plot(epochs, [v * 100 for v in best_history["val_cer"]], color="#DD8452", marker="s", label="Val CER (%)")

ax2.set_title("Best Model: Loss & Val CER (lr=1e-5, augment=True, 15 epochs)", fontsize=12)
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Train Loss", fontsize=11, color="#4C72B0")
ax3.set_ylabel("Val CER (%)", fontsize=11, color="#DD8452")
ax2.set_xticks(range(1, len(best_history["train_loss"]) + 1))
ax2.legend([line1, line2], ["Train Loss", "Val CER (%)"], fontsize=10)

plt.tight_layout()
plt.savefig("../results/figures/loss_curve_best.png", dpi=150)
plt.show()
print("Saved to results/figures/loss_curve_best.png")

In [ ]:
import json
import matplotlib.pyplot as plt
import os

# Load all history files
def load_history(filename):
    with open(f"results/metrics/{filename}", "r") as f:
        return json.load(f)

history_1 = load_history("lr=5e-5_augTrue_5ep.json")
history_2 = load_history("lr=1e-5_augTrue_5ep.json")
history_3 = load_history("lr=1e-4_augTrue_5ep.json")
history_4 = load_history("lr=5e-5_augFalse_5ep.json")

# Plot 2 — Val CER across all experiments
plt.figure(figsize=(10, 5))
plt.plot(range(1, 6), history_1["val_cer"], marker="o", label="lr=5e-5, augment=True")
plt.plot(range(1, 6), history_2["val_cer"], marker="o", label="lr=1e-5, augment=True")
plt.plot(range(1, 6), history_3["val_cer"], marker="o", label="lr=1e-4, augment=True")
plt.plot(range(1, 6), history_4["val_cer"], marker="o", label="lr=5e-5, augment=False")
plt.xlabel("Epoch")
plt.ylabel("Val CER")
plt.title("Validation CER Across All Experiments")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/val_cer_all_experiments.png", dpi=150)
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt
import os

# Load all history files
def load_history(filename):
    with open(f"results/metrics/{filename}", "r") as f:
        return json.load(f)

history_1 = load_history("lr=5e-5_augTrue_5ep.json")
history_2 = load_history("lr=1e-5_augTrue_5ep.json")
history_3 = load_history("lr=1e-4_augTrue_5ep.json")
history_4 = load_history("lr=5e-5_augFalse_5ep.json")
history_best = load_history("lr=1e-5_augTrue_15ep.json")

# Plot 2 — Val CER across all experiments
plt.figure(figsize=(10, 5))
plt.plot(range(1, 6), history_1["val_cer"], marker="o", label="lr=5e-5, augment=True")
plt.plot(range(1, 6), history_2["val_cer"], marker="o", label="lr=1e-5, augment=True")
plt.plot(range(1, 6), history_3["val_cer"], marker="o", label="lr=1e-4, augment=True")
plt.plot(range(1, 6), history_4["val_cer"], marker="o", label="lr=5e-5, augment=False")
plt.plot(range(1, 16), history_best["val_cer"], marker="o", label="lr=1e-5, 15 epochs", linewidth=2, linestyle="--", color="black")
plt.xlabel("Epoch")
plt.ylabel("Val CER")
plt.title("Validation CER Across All Experiments")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/val_cer_all_experiments(2).png", dpi=150)
plt.show()